# Matched heterocycle attachment-position benchmark

This notebook tests whether COAF distinguishes attachment-position changes more strongly than standard ECFP. It uses two mutually exclusive pair classes:

1. **Same scaffold, different position:** isolates a change in attachment position while holding the canonical scaffold constant. This is the primary orientation-sensitive comparison.
2. **Same position, different scaffold:** holds the labeled attachment position constant while changing the chemical scaffold. This is the chemical-change control.

Pairs that differ in both scaffold and position are excluded. Four descriptors are compared: standard ECFP after replacing `[Hg]` with oxygen, ECFP with `[Hg]` retained, COAF, and the rooted path-based orientation-aware fingerprint (POAF). ECFP and COAF use radius/iteration depth 3; POAF uses a maximum path length of 6. All fingerprints contain 1,024 bits.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'coaf.py').exists():
    raise FileNotFoundError(
        'Start Jupyter in the COAF project folder containing coaf.py.'
    )
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import coaf
import validation.matched_attachment_positions as matched_attachment_module
matched_attachment_module = importlib.reload(matched_attachment_module)

MatchedAttachmentConfig = matched_attachment_module.MatchedAttachmentConfig
load_matched_attachment_dataset = matched_attachment_module.load_matched_attachment_dataset
run_matched_attachment_benchmark = matched_attachment_module.run_matched_attachment_benchmark

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)

## 1. Input and output locations

The output folder is deliberately run-specific. The analysis refuses to overwrite a nonempty result directory unless `overwrite=True` is supplied explicitly. Change `RUN_NAME` when repeating the analysis with different parameters.

In [ ]:
DATA_FILE = PROJECT_DIR / 'data' / 'heterocycles.csv'
RUN_NAME = 'matched_attachment_positions_all_descriptors_r3_bits1024'
RESULT_DIR = PROJECT_DIR / 'results' / RUN_NAME
FIGURE_DIR = RESULT_DIR / 'figures'

print(f'Input:  {DATA_FILE}')
print(f'Output: {RESULT_DIR}')

## 2. Load and validate the dataset

The loader removes only rows for which all three required fields are blank. It rejects partially blank rows, invalid structures, and structures without exactly one `[Hg]` marker. Scaffold and complete-structure SMILES are canonicalized before pair groups are assigned. Symmetry-equivalent records remain visible in the audit but chemically identical canonical structures are excluded from the pair analysis.

In [ ]:
dataset = load_matched_attachment_dataset(DATA_FILE)

dataset_audit = pd.DataFrame({
    'metric': [
        'Valid molecules',
        'Completely blank rows removed',
        'Unique canonical scaffolds',
        'Unique attachment positions',
    ],
    'value': [
        len(dataset.frame),
        dataset.n_blank_rows_removed,
        dataset.frame['canonical_scaffold'].nunique(),
        dataset.frame['Position'].nunique(),
    ],
})
display(dataset_audit)
display(dataset.frame.head(10))

print('Molecules at each labeled attachment position')
display(
    dataset.frame['Position'].value_counts().sort_index()
    .rename_axis('Position').rename('n_molecules').to_frame()
)

## 3. Calculate fingerprints and construct matched pair groups

The calculation enumerates all unique molecule pairs but retains only the two prespecified pair classes. For every retained pair, it records Tanimoto distances for standard ECFP, ECFP-Hg, COAF, and POAF. Each attachment-aware descriptor is compared with standard ECFP using $\Delta d=d_{\mathrm{descriptor}}-d_{\mathrm{ECFP}}$. Positive values mean the indicated descriptor considers the pair more dissimilar than standard ECFP.

In [ ]:
config = MatchedAttachmentConfig(
    n_bits=1024,
    radius=3,
    poaf_max_path_length=6,
    bootstrap_replicates=2000,
    random_seed=123,
    ranking_size=20,
)

results = run_matched_attachment_benchmark(
    dataset,
    config=config,
    output_dir=RESULT_DIR,
)

print('Fingerprint audit')
display(results.fingerprint_summary.round(4))

print('Pair-group summary')
display(results.group_summary.round(4))

## 4. Paired descriptor-versus-ECFP distance plots

Each point represents the same molecular pair evaluated with standard ECFP on the x-axis and COAF, POAF, or ECFP-Hg on the y-axis. Points above the identity line have a larger distance with the comparison descriptor. Selective displacement above the identity line for the same-scaffold/different-position group would support attachment-position sensitivity rather than a general increase in descriptor distances.

In [ ]:
def save_figure(fig, filename, png_dpi=600):
    """Save a quantitative figure as editable SVG and high-resolution PNG."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    svg_path = FIGURE_DIR / f'{stem}.svg'
    png_path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=png_dpi, bbox_inches='tight')
    print(f'Saved figure: {svg_path}')
    print(f'Saved figure: {png_path}')
    return {'svg': svg_path, 'png': png_path}

def plot_group_distance_scatter(pairwise):
    groups = [
        ('same_scaffold_different_position',
         'Same scaffold, different attachment position'),
        ('same_position_different_scaffold',
         'Same attachment position, different scaffold'),
    ]
    comparisons = [('COAF', '#2563eb'), ('POAF', '#7c3aed'),
                   ('ECFP_Hg', '#059669')]
    fig, axes = plt.subplots(2, 3, figsize=(14, 9), sharex=True, sharey=True)
    for row, (group, group_title) in enumerate(groups):
        values = pairwise.loc[pairwise['pair_group'] == group]
        for column, (descriptor, color) in enumerate(comparisons):
            ax = axes[row, column]
            ax.scatter(
                values['ECFP_distance'], values[f'{descriptor}_distance'],
                s=16, alpha=0.30, color=color, edgecolors='none',
            )
            ax.plot([0, 1], [0, 1], '--', color='black', linewidth=1)
            ax.set(xlim=(0, 1), ylim=(0, 1), aspect='equal')
            if row == 0:
                ax.set_title(f'{descriptor} versus ECFP')
            if row == 1:
                ax.set_xlabel('Standard ECFP Tanimoto distance')
            if column == 0:
                ax.set_ylabel(
                    f'{group_title}\n$n$ = {len(values):,} pairs\n'
                    f'{descriptor} Tanimoto distance'
                )
            else:
                ax.set_ylabel(f'{descriptor} Tanimoto distance')
    fig.tight_layout()
    return fig

fig = plot_group_distance_scatter(results.pairwise_comparison)
save_figure(fig, 'orientation_aware_descriptors_vs_ECFP_by_pair_group.svg');

## 5. Direct comparison of descriptor differences

The scatter plots show both absolute distances, whereas the distributions below isolate the paired descriptor difference. A positive shift specifically for the same-scaffold/different-position group is the central expected result. The curves are Gaussian kernel-density estimates using Scott's bandwidth rule and each integrates to approximately one.

In [ ]:
def plot_descriptor_difference_distributions(
    pairwise,
    comparisons,
    figure_title,
):
    """
    Plot descriptor-minus-ECFP distributions for the two pair groups.

    Each comparison entry is:
        (descriptor name, color)
    """
    from scipy.stats import gaussian_kde

    groups = [
        (
            'same_scaffold_different_position',
            'Same scaffold, different position',
        ),
        (
            'same_position_different_scaffold',
            'Same position, different scaffold',
        ),
    ]

    difference_columns = [
        f'{descriptor}_minus_ECFP_distance'
        for descriptor, _ in comparisons
    ]

    pooled_values = (
        pairwise[difference_columns]
        .to_numpy(dtype=float)
        .ravel()
    )

    pooled_values = pooled_values[
        np.isfinite(pooled_values)
    ]

    margin = max(
        0.02,
        0.04 * np.ptp(pooled_values),
    )

    grid = np.linspace(
        pooled_values.min() - margin,
        pooled_values.max() + margin,
        500,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 4.8),
        sharex=True,
        sharey=True,
    )

    for ax, (pair_group, panel_title) in zip(
        axes,
        groups,
    ):
        subset = pairwise.loc[
            pairwise['pair_group'] == pair_group
        ]

        for descriptor, color in comparisons:
            values = subset[
                f'{descriptor}_minus_ECFP_distance'
            ].dropna().to_numpy(dtype=float)

            density = gaussian_kde(
                values,
                bw_method='scott',
            )(grid)

            ax.plot(
                grid,
                density,
                linewidth=2,
                color=color,
                label=descriptor,
            )

        ax.axvline(
            0,
            linestyle='--',
            color='black',
            linewidth=1,
            label='Equal to standard ECFP',
        )

        ax.set(
            xlabel='Descriptor distance − standard ECFP distance',
            title=panel_title,
        )

    axes[0].set_ylabel('Probability density')

    # One common legend is sufficient because both panels contain
    # the same descriptors.
    handles, labels = axes[1].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.04),
        ncol=len(handles),
        frameon=False,
    )

    fig.suptitle(
        figure_title,
        y=1.10,
        fontsize=13,
    )

    fig.tight_layout()

    return fig


# Figure 1: COAF and POAF together.
fig = plot_descriptor_difference_distributions(
    results.pairwise_comparison,
    comparisons=[
        ('COAF', '#2563eb'),
        ('POAF', '#7c3aed'),
    ],
    figure_title=(
        'Orientation-aware descriptor differences '
        'relative to standard ECFP'
    ),
)

save_figure(
    fig,
    'COAF_POAF_differences_from_ECFP_by_pair_group.svg',
)


# Figure 2: ECFP-Hg separately.
fig = plot_descriptor_difference_distributions(
    results.pairwise_comparison,
    comparisons=[
        ('ECFP_Hg', '#059669'),
    ],
    figure_title=(
        'Effect of retaining the attachment marker in ECFP'
    ),
)

save_figure(
    fig,
    'ECFP_Hg_differences_from_ECFP_by_pair_group.svg',
)

## 6. Probability-contour comparison of the two pair groups

The two matched-pair populations are shown together for each descriptor. Contours enclose fixed fractions of each population's KDE probability mass.


In [ ]:
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde


PAIR_GROUP_STYLES = {
    'same_scaffold_different_position': {
        'label': 'Same scaffold, different position',
        'color': '#2563eb',
    },
    'same_position_different_scaffold': {
        'label': 'Same position, different scaffold',
        'color': '#f97316',
    },
}


def kde_probability_thresholds(
    density,
    probability_masses,
):
    """
    Calculate density thresholds enclosing specified fractions of the
    normalized KDE probability mass.
    """
    ordered_density = np.sort(
        density.ravel()
    )[::-1]

    cumulative_mass = np.cumsum(
        ordered_density
    )

    cumulative_mass /= cumulative_mass[-1]

    thresholds = {}

    for mass in probability_masses:
        index = min(
            np.searchsorted(
                cumulative_mass,
                mass,
            ),
            len(ordered_density) - 1,
        )

        thresholds[mass] = (
            ordered_density[index]
        )

    return thresholds


def add_pair_group_probability_contours(
    ax,
    pairwise,
    descriptor,
    probability_masses=(0.50, 0.80, 0.95),
    grid_size=180,
    show_points=False,
):
    """
    Add independently normalized probability contours for both pair
    populations to one ECFP-versus-descriptor panel.
    """
    grid = np.linspace(
        0,
        1,
        grid_size,
    )

    grid_x, grid_y = np.meshgrid(
        grid,
        grid,
    )

    evaluation_points = np.vstack([
        grid_x.ravel(),
        grid_y.ravel(),
    ])

    line_styles = {
        0.50: '-',
        0.80: '--',
        0.95: ':',
    }

    line_widths = {
        0.50: 2.2,
        0.80: 1.8,
        0.95: 1.5,
    }

    for pair_group, style in (
        PAIR_GROUP_STYLES.items()
    ):
        group = pairwise.loc[
            pairwise['pair_group']
            == pair_group
        ]

        x = group[
            'ECFP_distance'
        ].to_numpy(dtype=float)

        y = group[
            f'{descriptor}_distance'
        ].to_numpy(dtype=float)

        if show_points:
            ax.scatter(
                x,
                y,
                s=8,
                alpha=0.08,
                color=style['color'],
                edgecolors='none',
                rasterized=True,
            )

        kde = gaussian_kde(
            np.vstack([x, y]),
            bw_method='scott',
        )

        density = kde(
            evaluation_points
        ).reshape(grid_x.shape)

        thresholds = (
            kde_probability_thresholds(
                density,
                probability_masses,
            )
        )

        for mass in probability_masses:
            contour = ax.contour(
                grid_x,
                grid_y,
                density,
                levels=[
                    thresholds[mass]
                ],
                colors=[
                    style['color']
                ],
                linestyles=[
                    line_styles[mass]
                ],
                linewidths=[
                    line_widths[mass]
                ],
            )

            ax.clabel(
                contour,
                fmt={
                    thresholds[mass]:
                    f'{int(100 * mass)}%'
                },
                inline=True,
                fontsize=8,
                colors=[
                    style['color']
                ],
            )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle='--',
        color='black',
        linewidth=1,
        alpha=0.8,
    )

    ax.set(
        xlim=(0, 1),
        ylim=(0, 1),
        aspect='equal',
        xlabel=(
            'Standard ECFP '
            'Tanimoto distance'
        ),
        ylabel=(
            f'{descriptor} '
            'Tanimoto distance'
        ),
        title=(
            f'{descriptor} sensitivity to '
            'attachment-position and '
            'scaffold changes'
        ),
    )


def plot_single_descriptor_probability_contours(
    pairwise,
    descriptor,
    probability_masses=(0.50, 0.80, 0.95),
    show_points=False,
):
    """
    Create one probability-contour figure for a single descriptor
    relative to standard ECFP.
    """
    fig, ax = plt.subplots(
        figsize=(8.2, 5.4),
    )

    add_pair_group_probability_contours(
        ax,
        pairwise,
        descriptor,
        probability_masses=(
            probability_masses
        ),
        show_points=show_points,
    )

    population_handles = [
        Line2D(
            [0],
            [0],
            color=style['color'],
            linewidth=2.2,
            label=style['label'],
        )
        for style in (
            PAIR_GROUP_STYLES.values()
        )
    ]

    line_styles = {
        0.50: '-',
        0.80: '--',
        0.95: ':',
    }

    probability_handles = [
        Line2D(
            [0],
            [0],
            color='#525252',
            linestyle=(
                line_styles[mass]
            ),
            linewidth=1.8,
            label=(
                f'{int(100 * mass)}% '
                'probability region'
            ),
        )
        for mass in probability_masses
    ]

    identity_handle = Line2D(
        [0],
        [0],
        color='black',
        linestyle='--',
        linewidth=1,
        label='Identity line',
    )

    ax.legend(
        handles=(
            population_handles
            + probability_handles
            + [identity_handle]
        ),
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        frameon=False,
    )

    fig.tight_layout()

    return fig


def save_figure_all_formats(
    fig,
    filename_stem,
    png_dpi=600,
):
    """
    Save a Matplotlib figure as editable SVG, vector PDF, and
    high-resolution PNG.
    """
    FIGURE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_paths = {}

    for file_format in (
        'svg',
        'pdf',
        'png',
    ):
        output_path = (
            FIGURE_DIR
            / f'{filename_stem}.{file_format}'
        )

        save_options = {
            'format': file_format,
            'bbox_inches': 'tight',
        }

        if file_format == 'png':
            save_options['dpi'] = (
                png_dpi
            )

        fig.savefig(
            output_path,
            **save_options,
        )

        output_paths[file_format] = (
            output_path
        )

        print(
            f'Saved figure: '
            f'{output_path}'
        )

    return output_paths


# Generate, save, and display one figure per descriptor.
for descriptor in (
    'COAF',
    'POAF',
    'ECFP_Hg',
):
    fig = (
        plot_single_descriptor_probability_contours(
            results.pairwise_comparison,
            descriptor=descriptor,
            probability_masses=(
                0.50,
                0.80,
                0.95,
            ),

            # Change to True to show faint
            # individual data points.
            show_points=False,
        )
    )

    save_figure_all_formats(
        fig,
        filename_stem=(
            'pair_group_probability_'
            f'contours_{descriptor}'
        ),
        png_dpi=600,
    )

    plt.show()

## 7. Clustered uncertainty estimates

Individual pair observations are not independent because each molecule participates in multiple pairs. The bootstrap therefore resamples complete scaffolds for the attachment-position group and complete unordered scaffold pairs for the chemical-change control. The intervals quantify uncertainty in the mean and median paired descriptor difference without treating every molecular pair as an independent replicate.

In [ ]:
display(results.bootstrap_summary.round(4))

## 8. Representative descriptor differences

The ranked tables identify pairs with the largest positive and negative values of $\Delta d$. These examples are useful for chemical interpretation, but they are selected extremes and should not be treated as independent statistical evidence.

In [ ]:
rank_columns = [
    'pair_group_label', 'molecule_id_1', 'molecule_id_2',
    'position_1', 'position_2', 'ECFP_distance', 'ECFP_Hg_distance',
    'COAF_distance', 'POAF_distance', 'COAF_minus_ECFP_distance',
    'POAF_minus_ECFP_distance', 'ECFP_Hg_minus_ECFP_distance',
]
print('Largest COAF-specific distance increases')
display(results.ranked_coaf_gain[rank_columns].head(10).round(4))

print('Largest POAF-specific distance increases')
display(results.ranked_poaf_gain[rank_columns].head(10).round(4))

print('Largest ECFP-Hg-specific distance increases')
display(results.ranked_ecfp_hg_gain[rank_columns].head(10).round(4))

print('Largest ECFP-specific distance increases')
display(results.ranked_ecfp_gain[rank_columns].head(10).round(4))

### Structures of representative descriptor differences

Each row below is one molecular pair, shown side by side. The `[Hg]` atom is retained so that the root position remains visible. Separate panels show the largest positive distance differences relative to standard ECFP for COAF, POAF, and ECFP-Hg, followed by the pairs for which standard ECFP most strongly exceeds COAF. These depictions are displayed only in the notebook and are not saved as quantitative figure files.

In [ ]:
from IPython.display import SVG, Markdown
from rdkit import Chem
from rdkit.Chem import Draw

def display_ranked_structure_pairs(table, title, descriptor, n_pairs=6):
    """Display the two rooted structures in each selected pair side by side."""
    selected = table.head(n_pairs).reset_index(drop=True)
    molecules = []
    legends = []

    for pair_rank, row in selected.iterrows():
        difference_column = f'{descriptor}_minus_ECFP_distance'
        metrics = (
            f"ECFP={row['ECFP_distance']:.3f}; "
            f"{descriptor}={row[f'{descriptor}_distance']:.3f}; "
            f"Δd={row[difference_column]:+.3f}"
        )
        for member in (1, 2):
            smiles = row[f'SMILES_{member}']
            molecule = Chem.MolFromSmiles(smiles)
            if molecule is None:
                raise ValueError(f'Could not depict SMILES: {smiles}')
            molecules.append(molecule)
            legends.append(
                f"Pair {pair_rank + 1}: {row[f'molecule_id_{member}']} "
                f"(position {row[f'position_{member}']})\n{metrics}"
            )

    display(Markdown(f'**{title}**'))
    drawing = Draw.MolsToGridImage(
        molecules,
        molsPerRow=2,
        subImgSize=(430, 280),
        legends=legends,
        useSVG=True,
    )
    # RDKit versions differ: some return an SVG string, while others
    # return an IPython SVG display object directly.
    if isinstance(drawing, (str, bytes)):
        display(SVG(data=drawing))
    else:
        display(drawing)

display_ranked_structure_pairs(
    results.ranked_coaf_gain,
    'Largest COAF-specific distance increases',
    descriptor='COAF',
    n_pairs=6,
)

display_ranked_structure_pairs(
    results.ranked_poaf_gain,
    'Largest POAF-specific distance increases',
    descriptor='POAF',
    n_pairs=6,
)

display_ranked_structure_pairs(
    results.ranked_ecfp_hg_gain,
    'Largest ECFP-Hg-specific distance increases',
    descriptor='ECFP_Hg',
    n_pairs=6,
)

display_ranked_structure_pairs(
    results.ranked_ecfp_gain,
    'Largest standard-ECFP distance increases relative to COAF',
    descriptor='COAF',
    n_pairs=6,
)

## 9. Interpretation guide

The strongest evidence for selective attachment-position sensitivity would consist of three concordant observations:

- same-scaffold/different-position points preferentially fall above the identity line;
- their $\Delta d$ distribution is shifted positively relative to the same-position/different-scaffold control; and
- the clustered confidence interval for the attachment-position effect excludes zero.

If both groups shift upward similarly for a descriptor, it may simply generate larger distances in this chemical series. If neither group shifts, the dataset does not present a strong root-relative discrimination challenge for that representation at the selected fingerprint depth. ECFP-Hg tests whether retaining the marker alone is sufficient, whereas comparison of COAF with POAF tests circular versus path-based root-relative encoding.